#Data Parsing

In [6]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\samba\AppData\Local\Temp\ipykernel_35564\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [7]:
pdf_loader = PyPDFLoader("D:\Practise_files\llama2-research-paper.pdf")

In [8]:
pdf_data = pdf_loader.load()

In [9]:
pdf_data[0].metadata

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2023-07-20T00:30:36+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2023-07-20T00:30:36+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': 'D:\\Practise_files\\llama2-research-paper.pdf',
 'total_pages': 77,
 'page': 0,
 'page_label': '1'}

In [10]:
pdf_data[0].page_content

'Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev\nPunit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich\nYinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Pushkar Mishra\nIgor Molybog Yixin Nie Andrew Poulton Jeremy Reizenstein Rashi Rungta Kalyan Saladi\nAlan Schelten Ruan Silva Eric Michael Smith Ranjan Subramanian Xiaoqing Ellen Tan Binh Tang\nRoss Taylor Adina Williams Jian Xiang Kuan Puxin Xu Zheng Yan Iliyan Zarov Yuchen Zhang\nAngela Fan Melanie Kambadur Sharan Narang Aurelien Rodriguez Robert Stojnic\

#Chunking

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [12]:
chunker = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

In [13]:
chunked_data = chunker.split_documents(pdf_data)

#embedding Model

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings

In [15]:
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5715.44it/s]


#Embeddings can store on vector DB, so we need vector DB

In [16]:
import faiss
from langchain_community.vectorstores import FAISS 
from langchain_community.docstore.in_memory import InMemoryDocstore

In [17]:
len(embeddings.embed_query("Hello world"))

384

In [18]:
index=faiss.IndexFlatL2(384)

In [19]:
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

In [20]:
vector_store.add_documents(chunked_data)

['dd21b375-c1fa-49a9-8ede-2a89aa7d4dfd',
 '670bf9f2-324e-46aa-9590-e8fb94c8487d',
 '1b718670-4485-4c54-a4ed-d602375d54da',
 '579c367b-766d-4b5e-9ca7-f5a481e312c6',
 '806921cc-021a-4840-a7b3-1daeda6f87cf',
 '92b89eba-43ff-4160-b630-aed2fd636439',
 'be7226df-dac4-47c8-b037-b62ac1499088',
 '40e2eb2e-80d9-43f6-a9ef-63877888b347',
 'ea75e84a-2145-4fc9-8e81-eaf380c1381c',
 '15b0af07-36ce-4097-aa9e-a421a56a3650',
 'a988648f-aa2e-4dc0-a3c0-00ad69c697c2',
 '968956fe-7444-4522-bdbb-47e288fd8826',
 '3941969b-dee1-4f9d-adb3-c6fb00a27190',
 'a52665c8-e80c-4943-90f9-ecb3d2b1fdd1',
 '83697c5f-05b2-4977-90ca-760685a04c77',
 'cffdd66a-c69e-4b93-b2d3-40b2650cf047',
 '8bde502e-c77e-4bac-a30a-36abb90b78e2',
 'e2ad6da0-4b85-4088-8247-7036b2f2b22e',
 '58050ffb-afea-46bb-8e00-24a66e482b4c',
 'e2d18e46-326f-473f-826c-fbb698d8dabc',
 '53b5fa08-b7e9-4864-97a2-81c6afcb704c',
 '48bd4b27-32d0-47fe-850d-b61573acdd7e',
 '943f19b0-83a0-41ad-9b12-ef8560d422cf',
 'e7468e89-5293-48fc-9777-e7db3a31be16',
 '9415008b-41c9-

In [21]:
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [22]:
retriever.invoke("What is the main topic of the research paper?")

[Document(id='a81a3670-1ac3-4ef8-8a46-eab4713f2d53', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\Practise_files\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 22, 'page_label': '23'}, page_content='activities(e.g., terrorism, theft, human trafficking);hateful and harmful activities(e.g., defamation, self-\nharm, eating disorders, discrimination); andunqualified advice(e.g., medical advice, financial advice, legal\n23'),
 Document(id='ca91d321-e437-4a70-9e13-feefa941a77d', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.f

In [23]:
from langchain_core.prompts import PromptTemplate

In [25]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

In [26]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [27]:
prompt = PromptTemplate.from_template(template)

In [33]:
from langchain_core.runnables import RunnablePassthrough

In [29]:
from langchain_core.output_parsers import StrOutputParser

In [36]:
import os
GOOGLE_API_KEY = os.getenv ("GOOGLE_API_KEY")  # This will return None if the environment variable is not set
if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [38]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [40]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [41]:
rag_chain ={"context": retriever |format_docs,"question": RunnablePassthrough()}|prompt|llm|StrOutputParser()

In [42]:
rag_chain.invoke("What is the main topic of the research paper?")

'Based on the provided context, the main topic of the research paper is the red teaming of AI models to identify and mitigate safety risks, specifically focusing on how models can be manipulated to produce problematic content and how to develop safer models.'